# ਪਾਠ 18: ਕ੍ਰਿਪਟੋਗ੍ਰਾਫਿਕ ਰਸੀਦਾਂ ਨਾਲ AI ਏਜੰਟਾਂ ਨੂੰ ਸੁਰੱਖਿਅਤ ਕਰਨਾ

## ਪ੍ਰਯੋਗਾਤਮਕ ਨੋਟਬੁੱਕ

ਇਹ ਨੋਟਬੁੱਕ ਚਾਰ ਅਭਿਆਸਾਂ ਰਾਹੀਂ ਲੰਘਦਾ ਹੈ:

1. ਇੱਕ ਏਜੰਟ ਟੂਲ ਕਾਲ ਲਈ ਆਪਣੀ ਪਹਿਲੀ ਰਸੀਦ **ਸਾਈਨ ਕਰੋ** ਅਤੇ ਇਸ ਦੀ ਪੁਸ਼ਟੀ ਕਰੋ।
2. ਰਸੀਦ ਨਾਲ **ਚੈਲ ਉੱਡਾਓ** ਅਤੇ ਪੁਸ਼ਟੀ ਅਸਫਲ ਹੋਣ ਨੂੰ ਦੇਖੋ।
3. ਇੱਕ ਤਿੰਨ-ਰਸੀਦ ਦਾ ਸਿੜਾ **ਬਣਾਓ** ਅਤੇ ਸਿੜੇ ਦੀ ਸਥਿਰਤਾ ਦੀ ਪੁਸ਼ਟੀ ਕਰੋ।
4. ਇੱਕ Microsoft ਏਜੰਟ ਫਰੇਮਵਰਕ ਟੂਲ ਕਾਲ ਨੂੰ **ਵਰੈਪ ਕਰੋ** ਤਾਂ ਜੋ ਹਰ ਕਾਰਵਾਈ ਇੱਕ ਰਸੀਦ ਜਾਰੀ ਕਰੇ।

ਸਾਰੇ ਕ੍ਰਿਪਟੋਗ੍ਰਾਫਿਕ ਪ੍ਰਿਮਿਟਿਵਜ਼ ਚੰਗੀ ਤਰ੍ਹਾਂ ਸੰਭਾਲੇ ਗਏ ਲਾਇਬਰੇਰੀਜ਼ ਤੋਂ ਆਏ ਹਨ (`pynacl` Ed25519 ਲਈ, `jcs` RFC 8785 ਕੈਨੋਨੀਕਲ JSON ਲਈ, `hashlib` SHA-256 ਲਈ Python ਮਿਆਰੀ ਲਾਇਬਰੇਰੀ ਤੋਂ)। ਰਸੀਦ ਲੌਜਿਕ ਖੁਦ ਸਾਫ ਸਧਾਰਣ Python ਹੈ ਜੋ ਤੁਸੀਂ ਪੜ੍ਹ ਅਤੇ ਬਦਲ ਸਕਦੇ ਹੋ।

ਸੈੱਲਾਂ ਨੂੰ ਕ੍ਰਮਵਾਰ ਚਲਾਓ। ਹਰ ਸੈਕਸ਼ਨ ਛੋਟਾ ਅਤੇ ਸਵੈ-ਨਿਰਭਰਸ਼ੀਲ ਹੈ।


## ਸੈਟਅਪ

ਦੋਨੋਂ ਡਿਪੈਂਡੈਂਸੀਜ਼ ਨੂੰ ਇੰਸਟਾਲ ਕਰੋ। ਦੋਹਾਂ ਕੋਲ ਪਰਮਿਸਿਵ ਲਾਇਸੰਸ ਹਨ (Apache-2.0 / MIT)।


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## ਸਹਾਇਕ ਯੂਟਿਲਿਟੀਜ਼

ਇਹ ਦੋ ਸਹਾਇਕ ਬੇਸ64url ਇੰਕੋਡਿੰਗ (ਬਿਨਾਂ ਪੈਡਿੰਗ ਦੇ) ਅਤੇ ਮਨਮਾਨੇ ਵਸਤੂਆਂ ਦੀ SHA-256 ਹੈਸ਼ਿੰਗ ਨੂੰ ਸੰਭਾਲਦੇ ਹਨ। ਇਹ ਨੋਟਬੁੱਕ ਦੇ ਬਾਕੀ ਹਿੱਸੇ ਨੂੰ ਸਬੰਧਿਤ ਲੋਜਿਕ 'ਤੇ ਕੇਂਦ੍ਰਿਤ ਰੱਖਦੇ ਹਨ।


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## ਭਾਗ 1: ਆਪਣਾ ਪਹਿਲਾ ਰਸੀਦ ਸਾਈਨ ਕਰੋ

ਸੋਚੋ ਕਿ ਸਾਡੇ ਏਜੰਟ **Contoso Travel** ਨੇ ਸਿਡਨੀ ਤੋਂ ਲੋਸ ਐਂਜਲਿਸ ਤੱਕ ਸਫਰ ਕਰਦੇ ਗਾਹਕ ਲਈ ਫਲਾਈਟਾਂ ਦੀ ਖੋਜ ਕੀਤੀ ਹੈ। ਅਸੀਂ ਇਸ ਟੂਲ ਕਾਲ ਨਾਲ ਸਬੰਧਿਤ ਗੇਟਵੇ ਦਾ ਬਿਆਨ ਦਰਜ ਕਰਨਾ ਚਾਹੁੰਦੇ ਹਾਂ ਤਾਂ ਜੋ ਭਵਿੱਖ ਦੇ ਆਡੀਟਰ ਇਸਨੂੰ ਅਲੱਗ ਤੌਰ 'ਤੇ ਪਿਨ ਕੀਤੇ ਹੋਏ ਜਾਰੀ ਕਰਨ ਵਾਲੇ ਕੁੰਜੀ ਹੇਠਾਂ ਤਸਦੀਕ ਕਰ ਸਕਣ।

### ਕਦਮ 1.1: ਸਾਈਨ ਕਰਨ ਲਈ ਕੁੰਜੀ ਬਣਾਓ

ਉਤਪਾਦਨ ਵਿੱਚ, ਏਜੰਟ ਦੀ ਸਾਈਨਿੰਗ ਕੁੰਜੀ ਇੱਕ ਹਾਰਡਵੇਅਰ ਸੁਰੱਖਿਆ ਮਾਡਿਊਲ (HSM), ਅਜੂਰੇ ਕੀ ਵੋਲ্ট ਜਾਂ ਸਮਾਨ ਸੁਰੱਖਿਅਤ ਸਟੋਰ ਵਿੱਚ ਰਹਿੰਦੀ। ਇਸ ਪਾਠ ਲਈ ਅਸੀਂ ਯਾਦاشت ਵਿੱਚ ਇੱਕ ਨਵੀਂ ਕੁੰਜੀ ਤਿਆਰ ਕਰਦੇ ਹਾਂ।


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### ਕਦਮ 1.2: ਰਸੀਦ ਪੇਲੋਡ ਬਣਾਓ

ਪੇਲੋਡ ਵਿੱਚ ਉਹ ਸਭ ਕੁਝ ਹੁੰਦਾ ਹੈ ਜਿਸਦੀ ਅਸੀਂ ਚਾਹਣਾ ਹੈ ਕਿ ਰਸੀਦ ਪੁਸ਼ਟੀ ਕਰੇ: ਕਿਸ ਨੇ ਕੰਮ ਕੀਤਾ, ਕਿਹੜਾ ਟੂਲ, ਕਿਹੜੇ ਦਲੀਲਾਂ ਨਾਲ, ਕੀ ਪਰਤਿਆ, ਕਿਸ ਨੀਤੀ ਦੇ ਤਹਿੱਤ, ਅਤੇ ਕਦੋਂ। ਅਸੀਂ ਦਲੀਲਾਂ ਅਤੇ ਨਤੀਜੇ ਦਾ ਹੈਸ਼ ਕਰਦੇ ਹਾਂ ਨਾ ਕਿ ਉਹਨਾਂ ਨੂੰ ਸਿੱਧੇ ਸ਼ਾਮਲ ਕਰਦੇ ਹਾਂ ਤਾਂ ਜੋ ਰਸੀਦ ਸੰਵੇਦਨਸ਼ੀਲ ਸਮੱਗਰੀ ਲੀਕ ਨਾ ਕਰੇ।  


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### ਕਦਮ 1.3: ਰਸੀਦ 'ਤੇ ਦਸਤਖਤ ਕਰੋ ਅਤੇ ਇਕੱਠਾ ਕਰੋ

ਤਿੰਨ ਕਦਮ:

1. JCS ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਪੇਲੋਡ ਨੂੰ ਕੈਨੋਨੀਕਲ ਬਣਾਓ ਤਾਂ ਜੋ ਦੋ ਵਿਧੀਆਂ ਜੋ ਇੱਕੋ ਲਾਜ਼ਮੀ ਰਸੀਦ ਬਣਾਉਂਦੀਆਂ ਹਨ, ਬਾਈਟ-ਪਛਾਣਯੋਗ ਬਾਈਟ ਬਣਾਉਣ।
2. ਕੈਨੋਨੀਕਲ ਬਾਈਟਾਂ ਨੂੰ ਸਿੱਧਾ Ed25519 ਪ੍ਰਾਈਵੇਟ ਕੁੰਜੀ ਨਾਲ ਦਸਤਖਤ ਕਰੋ। PureEdDSA ਸੰਦੇਸ਼ ਨੂੰ ਅੰਦਰੂਨੀ ਤੌਰ 'ਤੇ ਹੈਸ਼ ਕਰਦਾ ਹੈ, ਇਸ ਲਈ ਇੱਕ ਵਾਧੂ ਪ੍ਰੀ-ਹੈਸ਼ ਪ੍ਰੋਟੋਕੋਲ ਨੂੰ ਬਦਲ ਦੇਵੇਗਾ।

ਫਿਰ ਦਸਤਖਤ ਮੂਲ ਪੇਲੋਡ ਨਾਲ ਜੁੜੀ ਹੈ ਤਾਂ ਜੋ ਅੰਤਿਮ ਰਸੀਦ ਤਿਆਰ ਹੋਵੇ।


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### ਕਦਮ 1.4: ਰਸੀਦ ਦੀ ਜਾਂਚ ਕਰੋ

ਜਾਂਚ ਪ੍ਰਕਿਰਿਆ ਨੂੰ ਉਲਟ ਦਿੰਦੀ ਹੈ। ਅਸੀਂ ਦਸਤਖਤ ਨੂੰ ਹਟਾਉਂਦੇ ਹਾਂ, ਕੈਨਨੀਕਲ ਬਾਈਟਸ ਨੂੰ ਦੁਬਾਰਾ ਗਣਨਾ ਕਰਦੇ ਹਾਂ, ਅਤੇ ਦਸਤਖਤ ਨੂੰ ਜਾਰੀ ਕਰਨ ਵਾਲੇ ਦੀ ਸਰਵਜਨਿਕ ਕੁੰਜੀ ਦੇ ਖਿਲਾਫ ਜਾਂਚਦੇ ਹਾਂ।

ਜਾਂਚ ਆਫਲਾਈਨ ਹੁੰਦੀ ਹੈ: ਕੋਈ ਸੇਵਾ ਕਾਲ ਕਰਨ ਦੀ ਜਰੂਰਤ ਨਹੀਂ ਅਤੇ ਕੋਈ ਨੈੱਟਵਰਕ ਪਹੁੰਚ ਨਹੀਂ। ਇਸਨੂੰ ਰਸੀਦ ਤੋਂ ਇਲਾਵਾ ਇੱਕ ਚੀਜ਼ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ: ਜਾਰੀ ਕਰਨ ਵਾਲੇ ਦੀ ਸਰਵਜਨਿਕ ਕੁੰਜੀ, ਜੋ ਇੱਕ ਐਸੇ ਚੈਨਲ ਰਾਹੀਂ ਪ੍ਰਾਪਤ ਕੀਤੀ ਜਾਂਦੀ ਹੈ ਜਿਸ 'ਤੇ ਆਡੀਟਰ ਪਹਿਲਾਂ ਹੀ ਭਰੋਸਾ ਕਰਦਾ ਹੈ, ਜਿਵੇਂ ਕਿ ਜਾਰੀ ਕਰਨ ਵਾਲਾ ਪ੍ਰਕਾਸ਼ਿਤ ਕਰਦਾ ਹੈ ਭਾਵ JWK ਸੈੱਟ ਜਾਂ ਪਹਿਲਾਂ ਪਿੰਨ ਕੀਤੀ ਗਈ کوئی ਵੈਲਯੂ। ਰਸੀਦ ਦੇ ਅੰਦਰ `public_key` ਫੀਲਡ ਉਸ ਭੂਮਿਕਾ ਨੂੰ ਨਿਭਾ ਨਹੀਂ ਸਕਦਾ, ਕਿਉਂਕਿ ਜੋ ਵੀ ਰਸੀਦ ਲਿਖਦਾ ਹੈ ਉਹ ਫੀਲਡ ਵੀ ਲਿਖਦਾ ਹੈ। ਸੈਕਸ਼ਨ 2 ਇਸ ਜਾਲਸਾਜ਼ੀ ਨੂੰ ਰੋਕਦਾ ਹੈ।


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


ਤੁਹਾਨੂੰ `Receipt is valid: True`, `Pre-hashed receipt valid: False` ਅਤੇ `Malformed receipts rejected: True` ਵੇਖਣ ਚਾਹੀਦੇ ਹਨ। ਸਕਾਰਾਤਮਕ ਮਾਮਲਾ ਸਿੱਧਾ-JCS ਸਾਈਨਿੰਗ ਰਾਹ ਕੰਮ ਕਰਦਾ ਹੈ ਇਹ ਸਾਬਤ ਕਰਦਾ ਹੈ; ਨਕਾਰਾਤਮਕ ਕੰਟਰੋਲ ਸਾਈਨਚਰ-ਸਕੋਪ ਨਿਯਮ ਨੂੰ ਪ੍ਰੋਜ਼ ਵਜੋਂ ਛੱਡਣ ਦੀ ਬਜਾਇ ਚਲਾਉਣਯੋਗ ਬਣਾਉਂਦਾ ਹੈ; ਆਖਰੀ ਲਾਈਨ ਵਿਗੜੇ ਹੋਏ ਇਨਪੁੱਟ ਨੂੰ ਸੁਤੰਤਰਤਾ ਵਜੋਂ ਉਠਾਉਣ ਦੀ ਬਜਾਇ `False` ਵਾਪਸ ਦਿਖਾਉਂਦੀ ਹੈ।


## ਭਾਗ 2: ਰਸੀਦ ਨਾਲ ਛੇੜਖਾਨੀ

ਰਸੀਦਾਂ ਦਾ ਸਾਰ ਇਹ ਹੈ ਕਿ ਉਹ ਛੇੜਛਾੜ ਸਮਝਦਾਰ ਹੋਣ। ਆਓ ਇਸਨੂੰ ਸਾਬਤ ਕਰੀਏ।

ਅਸੀਂ ਰਸੀਦ ਦੇ ਬਿਲਕੁਲ ਇੱਕ ਅੱਖਰ ਨੂੰ ਬਦਲਾਂਗੇ ਅਤੇ ਵੈਰੀਫਿਕੇਸ਼ਨ ਨਾਕਾਮ ਹੋਣ ਦਾ ਦੇਖਾਂਗੇ।


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### ਕੀ ਹੁਣੇ ਹੋਇਆ?

ਜਦੋਂ ਅਸੀਂ `policy_id` ਨੂੰ ਬਦਲਿਆ, ਤਾਂ canonical ਬਾਈਟਸ ਬਦਲ ਗਈਆਂ। ਸਿਗਨੇਚਰ (ਜੋ ਮੂਲ canonical ਬਾਈਟਸ 'ਤੇ ਸੀ) ਹੁਣ ਮੇਲ ਨਹੀਂ ਖਾਂਦੀ। ਜाँच ਸਹੀ ਤਰ੍ਹਾਂ `False` ਵਾਪਸ ਕਰਦੀ ਹੈ।

ਕੋਈ ਵੀ ਤਰੀਕਾ ਨਹੀਂ ਹੈ ਕਿ ਸਾਇਨ ਕੀਤੇ ਖੇਤਰ ਨੂੰ ਬਦਲਿਆ ਜਾਵੇ, ਜਿਸਦਾ ਮਤਲਬ ਹੈ ਕਿ `signature` ਆਬਜੈਕਟ ਦੇ ਬਾਹਰ ਕੁਝ ਵੀ ਹੋਵੇ, ਅਤੇ ਤਰਜੀਹਕਤਾ ਦੇ ਜਨਤਾ ਕੀ ਨਾਲ ਰਸੀਦ ਦੀ ਪੁਸ਼ਟੀ ਹੋਵੇ, ਜਦ ਤੱਕ ਕਿ ਹਮਲਾਵਰ ਕੋਲ ਜਨਤਾ ਦੀ ਪ੍ਰਾਈਵੇਟ ਕੁੰਜੀ ਨਾ ਹੋਵੇ। `signature` ਆਬਜੈਕਟ ਖੁਦ ਸਾਇਨ ਕੀਤਾ ਨਹੀਂ ਗਿਆ: `sig` ਸਿਗਨੇਚਰ ਹੈ, ਅਤੇ `verify_receipt` `public_key` ਨੂੰ ਅਣਡਿੱਠਾ ਕਰਦਾ ਹੈ, ਇਸਲਈ ਸਿਰਫ ਉਸ ਖੇਤਰ ਨੂੰ ਸੋਧਣਾ ਕੁਝ ਨਹੀਂ ਬਦਲਦਾ। ਜਦ ਤੱਕ ਪ੍ਰਾਈਵੇਟ ਕੀ ਇੱਕ ਕੀ ਵਾਲਟ ਵਿੱਚ ਹੈ ਅਤੇ ਪਰਖਣ ਵਾਲੇ ਜਨਤਾ ਦੁਆਰਾ ਸਵਤੰਤਰ ਤੌਰ 'ਤੇ ਪ੍ਰਕਾਸ਼ਿਤ ਕੀ ਵਰਤਦੇ ਹਨ, ਸਾਇਨ ਕੀਤੇ ਖੇਤਰ ਵਿੱਚ ਕੋਈ ਤਬਦੀਲੀ ਲੁਕਾਈ ਨਹੀਂ ਜਾ ਸਕਦੀ। ਇਹ ਸਥਿਤੀ ਅਹੰਕਾਰਪੂਰਨ ਹੈ: ਅਗਲਾ ਸੈੱਲ ਇੱਕ ਸੋਧੀ ਹੋਈ ਰਸੀਦ ਦਿਖਾਉਂਦਾ ਹੈ ਜੋ ਇੱਕ ਵੱਖਰੇ ਕੀ ਖਿਲਾਫ ਬਿਲਕੁਲ ਤੁਰੰਤ ਪੁਸ਼ਟੀ ਕਰਦੀ ਹੈ।

ਖੁਦ ਕੋਸ਼ਿਸ਼ ਕਰੋ: ਉਪਰ ਦਿੱਤੇ ਸੈੱਲ ਵਿੱਚ `tool_name` ਜਾਂ `agent_id` ਜਾਂ `timestamp` ਨੂੰ ਸੋਧੋ ਅਤੇ ਦੁਬਾਰਾ ਚਲਾਓ। ਸਾਇਨ ਕੀਤੇ ਖੇਤਰ ਵਿੱਚ ਹਰ ਤਬਦੀਲੀ ਇੱਕ ਅਵੈਧ ਰਸੀਦ ਤਿਆਰ ਕਰਦੀ ਹੈ।


### ਉਹ ਛੇੜਛਾੜ ਜੋ ਦਸਤਖ਼ਤ ਨਾ ਪਕੜੇ

ਉਪਰ ਦਿੱਤਾ ਗਿਆ ਹਮਲਾਵਰ ਮੁੜ-ਦਸਤਖ਼ਤ ਨਹੀਂ ਕਰਦਾ। ਇੱਕ ਹੋਰ ਧਿਆਨਧਾਰਪੂਰਣ ਹਮਲਾਵਰ ਇਹ ਕਰੇਗਾ: ਰਸੀਦ ਨੂੰ ਬਦਲੇਗਾ,
ਆਪਣੇ ਆਪ ਬਣਾਈ ਗਈ ਕੁੰਜੀ ਨਾਲ ਇਸ 'ਤੇ ਦਸਤਖ਼ਤ ਕਰੇਗਾ, ਅਤੇ ਉਹ ਕੁੰਜੀ
`public_key` ਖੇਤਰ ਵਿੱਚ ਰੱਖੇਗਾ। ਨਤੀਜਾ ਇਹ ਹੈ ਕਿ ਰਸੀਦ ਦੀ ਦਸਤਖ਼ਤ ਇਸਦੀ ਸਮੱਗਰੀ ਲਈ ਪੂਰੀ ਤਰ੍ਹਾਂ ਅਸਲੀ ਹੁੰਦੀ ਹੈ।


ਇਸਨੂੰ ਚਲਾਓ ਅਤੇ ਦੇਖੋ ਕਿ ਦੋਨਾਂ ਚੈਕਾਂ ਵਿੱਚੋਂ ਕਿਹੜਾ ਇਹਨੂੰ ਫੜਦਾ ਹੈ।


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


ਤੁਹਾਨੂੰ `True` ਫਿਰ `False` ਦੇਖਣਾ ਚਾਹੀਦਾ ਹੈ। ਉਹ ਤਿੰਨ `assert` ਲਾਈਂਜ਼ ਇਸ ਨੂੰ
ਇੱਕ ਚੈੱਕ ਬਣਾਉਂਦੀਆਂ ਹਨ: ਜੇਕਰ ਜਾਲਸਾਜ਼ੀ ਕਦੇ ਵੀ ਜਾਰੀ ਕਰਨ ਵਾਲੇ ਦੀ ਕੁੰਜੀ ਨਾਲ ਵੈਰੀਫਾਈ ਹੁੰਦੀ ਹੈ ਤਾਂ ਸੈੱਲ ਇੱਕ
ਗਲਤੀ ਉਠਾਉਂਦਾ ਹੈ।

ਜਾਲਸਾਜ਼ੀ ਅੰਦਰੂਨੀ ਤੌਰ 'ਤੇ ਸੰਗਤ ਹੈ। ਇਸ ਦਾ ਦਸਤਖ਼ਤ ਸੱਚਮੁਚ ਇਸ ਦੀ
ਸਮੱਗਰੀ ਅਤੇ ਜਿਸ ਕੁੰਜੀ ਨੂੰ ਇਹ ਲਿਆ ਹੈ, ਨਾਲ ਮਿਲਦਾ ਹੈ, ਇਸ ਲਈ ਇੱਕ ਵੇਰੀਫਾਇਰ ਜੋ ਕੁੰਜੀ ਪ੍ਰਾਪਤ ਕਰਦਾ ਹੈ
ਰਸੀਦ ਵਿਚੋਂ, ਇਸ ਨੂੰ ਵੈਧ ਦੱਸਦਾ ਹੈ। ਰਸੀਦ ਵਿੱਛ ਕੋਈ ਗਲਤ ਨਹੀਂ ਦਿੱਸਦਾ ਕਿਉਂਕਿ
ਇਸ ਵਿੱਚ ਕੁਝ ਵੀ ਤਰਕ-ਵਿਰੋਧੀ ਨਹੀਂ ਹੈ।

ਇਕੱਲੀ ਚੀਜ਼ ਜੋ ਇਸਨੂੰ ਫੜਦੀ ਹੈ ਉਹ ਹੈ ਇੱਕ ਕੁੰਜੀ ਨਾਲ ਚੈਕ ਕਰਨਾ ਜਿਸ ਉਤੇ ਹਮਲਾਵਰ ਨੇ ਕਿਸੇ ਤਰੀਕੇ ਨਾਲ ਪ੍ਰਭਾਵ ਨਹੀਂ ਪਾਇਆ।
ਇਹੀ ਕਰਕੇ `verify_receipt` ਕਿ ਕੁੰਜੀ ਨੂੰ ਪੈਰਾਮੀਟਰ ਵਜੋਂ ਲੈਂਦਾ ਹੈ: ਇੱਕ
ਫੰਕਸ਼ਨ ਜਿਸ ਦਾ ਦਸਤਖ਼ਤ `verify_receipt(receipt)` ਹੈ ਇਸ ਨੂੰ ਸੁਰੱਖਿਅਤ ਤਰੀਕੇ ਨਾਲ ਲਿਖਿਆ ਨਹੀਂ ਜਾ ਸਕਦਾ,
ਕਿਉਂਕਿ ਇਹ ਸਿਰਫ ਉਹੀ ਕੁੰਜੀ ਪ੍ਰਾਪਤ ਕਰ ਸਕਦਾ ਹੈ ਜੋ ਹਮਲਾਵਰ ਨੇ ਦਿੱਤੀ ਹੋਵੇ।

ਉਹੀ ਰਸੀਦ `sample_receipts/04_resigned_receipt.json` ਵਜੋਂ ਸੁਰੱਖਿਅਤ ਕੀਤੀ ਜਾਂਦੀ ਹੈ ਜੇ ਤੁਸੀਂ
ਇਸ ਨੂੰ ਨੋਟਬੁੱਕ ਤੋਂ ਬਾਹਰ ਜਾਂਚਣਾ ਚਾਹੁੰਦੇ ਹੋ।


## ਭਾਗ 3: ਰਸੀਦਾਂ ਨੂੰ ਇਕੱਠਾ ਜੋੜਨਾ

ਇੱਕ ਇੱਕਲੀਆਂ ਰਸੀਦ ਇੱਕ ਕਰਵਾਈ ਦੀ ਰੱਖਿਆ ਕਰਦੀ ਹੈ। ਜ਼ਿਆਦਾਤਰ ਏਜੰਟ ਕਈ ਕਰਵਾਈਆਂ ਕਰਦੇ ਹਨ। ਪੂਰੀ ਕ੍ਰਮਬੱਧਤਾ ਨੂੰ ਤਾਮਪਰ-ਸਬੂਤੀਯੋਗ ਬਣਾਉਣ ਲਈ, ਅਸੀਂ ਹਰ ਇੱਕ ਰਸੀਦ ਨੂੰ ਪਿਛਲੀ ਰਸੀਦ ਨਾਲ ਜੋੜਦੇ ਹਾਂ ਜਿਸ ਵਿੱਚ ਪਿਛਲੀ ਰਸੀਦ ਦੇ ਹੈਸ਼ ਨੂੰ ਨਵੀਂ ਰਸੀਦ ਦੇ ਪੇਲੋਡ ਵਿੱਚ ਸ਼ਾਮਲ ਕੀਤਾ ਜਾਂਦਾ ਹੈ।

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

ਜੇ ਕੋਈ ਵੀ ਪੇਸ਼ ਕੀਤੇ ਕ੍ਰਮ ਵਿੱਚੋਂ ਕਿਸੇ ਰਸੀਦ ਨੂੰ ਹਟਾ ਦਿੰਦਾ ਹੈ ਜਾਂ ਕ੍ਰਮ ਬਦਲਦਾ ਹੈ, ਤਾਂ ਅਗਲਾ ਜੋੜ ਤੋੜ ਜਾਂਦਾ ਹੈ। ਛੇਤੀ ਕੱਟਣਾ ਜਾਂ ਦਸਤਖਤ ਕਰਨ ਵਾਲੇ ਵੱਲੋਂ ਨਿਯੰਤਰਿਤ ਸੁੱਫਿਕਸ ਪੁਨਰਲੇਖਨ ਦਾ ਪਤਾ ਲਗਾਉਣ ਲਈ ਵੀ ਮੋੜ੍ਹੀ ਮੌਜੂਦਾ ਸਿਰ, ਬਾਹਰੀ ਚੈਕਪੋਇੰਟ ਜਾਂ ਪ੍ਰਮਾਣਿਤ ਆਬਾਦੀ ਬਧਨ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ।


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


ਹੁਣ ਵਿਚਕਾਰਲੇ ਰਸੀਦ ਨਾਲ чарਾਜੂੜ ਕਰਕੇ ਚੇਨ ਨੂੰ ਤੋੜੋ ਅਤੇ ਮੁੜ-ਸत्यਾਪਨ ਕਰੋ। ਤਬਦੀਲ ਕੀਤੀ ਗਈ ਰਸੀਦ ਆਪਣੇ ਸਿਗਨੇਚਰ ਜਾਂਚ ਵਿੱਚ ਨਾਕਾਮ ਰਹਿੰਦੀ ਹੈ, ਅਤੇ ਅਗਲੀ ਰਸੀਦ ਆਪਣੀ ਚੇਨ-ਲਿੰਕ ਜਾਂਚ ਵਿੱਚ ਨਾਕਾਮ ਰਹਿੰਦੀ ਹੈ (ਕਿਉਂਕਿ ਇਸ ਦਾ `previous_receipt_hash` ਹੁਣ ਤਬਦੀਲ ਕੀਤੀ ਗਈ ਵਿਚਕਾਰਲੀ ਰਸੀਦ ਦੇ ਹੈਸ਼ ਨਾਲ ਮੇਲ ਨਹੀਂ ਖਾਂਦਾ)।  


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


ਰਸੀਦ 0 ਅਜੇ ਵੀ ਵੈਰੀਫਾਈ ਹੁੰਦੀ ਹੈ (ਇਸ ਵਿੱਚ ਕੋਈ ਸੋਧ ਨਹੀਂ ਕੀਤੀ ਗਈ ਅਤੇ ਇਸ ਦਾ ਕੋਈ ਪੂਰਵਜ ਨਹੀਂ ਜਿਸ 'ਤੇ ਨਿਰਭਰ ਕੀਤਾ ਜਾ ਸਕੇ)। ਰਸੀਦ 1 ਆਪਣੀ ਦਸਤਖਤ ਜਾਂਚ ਵਿੱਚ ਫੇਲ ਹੁੰਦੀ ਹੈ ਕਿਉਂਕਿ ਅਸੀਂ `tool_args_hash` ਨੂੰ ਬਦਲ ਦਿੱਤਾ ਹੈ। ਰਸੀਦ 2 ਆਪਣੀ ਚੇਨ-ਲਿੰਕ ਜਾਂਚ ਵਿੱਚ ਫੇਲ ਹੁੰਦੀ ਹੈ ਕਿਉਂਕਿ ਇਸ ਦਾ `previous_receipt_hash` ਮੁਢਲੀ (ਹੁਣ-ਸੋਧੀ ਗਈ) ਰਸੀਦ 1 ਦੇ ਖਿਲਾਫ ਗਣਨਾ ਕੀਤਾ ਗਿਆ ਸੀ।

ਭਲੇ ਹੀ ਕੋਈ ਹਮਲਾਵਰ ਸੋਧੀ ਗਈ ਰਸੀਦ 1 ਨੂੰ ਦੁਬਾਰਾ ਦਸਤਖਤ ਕਰਦਾ ਹੈ, ਨਵੀਂ ਦਸਤਖਤ ਜਾਰੀ ਕਰਨ ਵਾਲੇ ਦੀ ਸਵਿੱਚੀ ਚਾਬੀ ਵਗੈਰਾ ਦੇ ਬਿਨਾਂ ਫੇਲ ਹੋ ਜਾਏਗੀ (ਜਿਵੇਂ ਕਿ ਸੈਕਸ਼ਨ 2 ਨੇ ਦਿਖਾਇਆ ਸੀ), ਅਤੇ ਰਸੀਦ 2 ਵਿੱਚ ਚੇਨ-ਲਿੰਕ ਦਾ ਬੇਮਿਲਾਪ ਤਬਦੀਲੀ ਨੂੰ ਉਜਾਗਰ ਕਰੇਗਾ। ਤਬਦੀਲੀ ਨੂੰ ਛੁਪਾਉਣ ਲਈ, ਹਮਲਾਵਰ ਨੂੰ ਸੋਧ ਦੇ ਬਿੰਦੂ ਤੋਂ ਅੱਗੇ ਹਰ ਰਸੀਦ ਨੂੰ ਦੁਬਾਰਾ ਦਸਤਖਤ ਕਰਨੀ ਪਵੇਗੀ, ਜਿਸ ਲਈ ਜਾਰੀ ਕਰਨ ਵਾਲੇ ਦੀ ਪ੍ਰਾਈਵੇਟ ਚਾਬੀ ਦੀ ਜ਼ਰੂਰਤ ਪਵੇਗੀ।


## ਭਾਗ 4: ਇੱਕ ਏਜੰਟ ਟੂਲ ਕਾਲ ਨੂੰ ਰਸੀਦ ਸਾਈਨਿੰਗ ਨਾਲ ਵ੍ਰੈਪ ਕਰੋ

ਇਕ ਅਸਲੀ ਤਾਇਨਾਤੀ ਵਿੱਚ, ਤੁਸੀਂ ਨਹੀਂ ਚਾਹੁੰਦੇ ਕਿ ਹਰ ਏਜੰਟ ਲੇਖਕ ਨੂੰ `make_receipt` ਕਾਲ ਕਰਨ ਦੀ ਯਾਦ ਹੋਵੇ। ਤੁਸੀਂ ਚਾਹੁੰਦੇ ਹੋ ਕਿ ਹਰ ਟੂਲ ਇੰਵੋਕੇਸ਼ਨ ਲਈ ਰਸੀਦ ਸਾਈਨਿੰਗ ਆਪਣੇ ਆਪ ਹੋ ਜਾਵੇ।

ਇੱਥੇ ਸਭ ਤੋਂ ਸਾਦਾ ਪੈਟਰਨ ਹੈ: ਇੱਕ ਵ੍ਰੈਪਰ ਕਲਾਸ ਜੋ ਕਿਸੇ ਵੀ ਕਾਲ ਕਰਨ ਯੋਗ ਟੂਲ ਫੰਕਸ਼ਨ ਨੂੰ ਲੈਂਦੀ ਹੈ ਅਤੇ ਇਸ ਦਾ ਇੱਕ ਰਸੀਦ ਜਾਰੀ ਕਰਨ ਵਾਲਾ ਵਰਜਨ ਵਾਪਸ ਕਰਦੀ ਹੈ। ਇਹ ਕਿਸੇ ਵੀ ਏਜੰਟ ਫਰੇਮਵਰਕ ਲਈ ਅਨੁਕੂਲ ਹੁੰਦਾ ਹੈ, ਜਿਸ ਵਿੱਚ ਮਾਇਕ੍ਰੋਸਾਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ (`agent_framework.foundry`) ਸ਼ਾਮਿਲ ਹੈ।

ਜੇ ਤੁਹਾਡੇ ਕੋਲ ਮਾਇਕ੍ਰੋਸਾਫਟ ਫਾਊਂਡਰੀ ਪ੍ਰੋਜੈਕਟ ਸੈੱਟਅਪ ਨਹੀਂ ਹੈ, ਤਾਂ ਹੇਠਾਂ ਦਿੱਤਾ ਲੋਕਲ ਮੌਕ ਫਿਰ ਵੀ ਪੈਟਰਨ ਦੀ ਪ੍ਰਦਰਸ਼ਨੀ ਕਰਦਾ ਹੈ।


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### ਮਾਇਕ੍ਰੋਸਾਫਟ ਏਜੰਟ ਫ੍ਰੇਮਵਰਕ ਨਾਲ ਏਕਤਾ ਕਰਨਾ

ਉਪਰੋਕਤ `ReceiptedTool` ਰੈਪਰ ਫ੍ਰੇਮਵਰਕ-ਅਗਨੌਸਟਿਕ ਹੈ। ਇਸਨੂੰ ਮਾਇਕ੍ਰੋਸਾਫਟ ਏਜੰਟ ਫ੍ਰੇਮਵਰਕ ਨਾਲ ਬਣਾਏ ਗਏ ਏਜੰਟ ਦੇ ਅੰਦਰ ਵਰਤਣ ਲਈ, ਰੈਪ ਕੀਤੀ ਗਈ ਫੰਕਸ਼ਨ ਨੂੰ ਇੱਕ ਟੂਲ ਵਜੋਂ ਰਜਿਸਟਰ ਕਰੋ। ਇੱਕ ਖਾਕਾ (ਤੁਸੀਂ ਮੌਕ ਨੂੰ ਅਸਲੀ ਮਾਇਕ੍ਰੋਸਾਫਟ ਫਾਊਂਡਰੀ ਟੂਲ ਰਜਿਸਟ੍ਰੇਸ਼ਨ ਨਾਲ ਬਦਲੋਂਗੇ):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

ਏਜੰਟ ਫ੍ਰੇਮਵਰਕ ਨੂੰ ਰਸੀਦਾਂ ਬਾਰੇ ਕੁਝ ਜਾਣਕਾਰੀ ਦੀ ਲੋੜ ਨਹੀਂ। ਰਸੀਦ ਸਾਈਨਿੰਗ ਟੂਲ ਦੇ ਆਲੇ ਦੁਆਲੇ ਲਪੇਟਿਆ ਜਾਂਦਾ ਹੈ, ਫ੍ਰੇਮਵਰਕ ਵਿੱਚ ਨਹੀਂ ਜੋੜਿਆ ਜਾਂਦਾ। ਇਸ ਤਰ੍ਹਾਂ ਤੁਸੀਂ ਮੌਜੂਦਾ ਏਜੰਟ ਕੋਡ ਵਿੱਚ ਪ੍ਰੂਵਨੈਂਸ ਸ਼ਾਮਲ ਕਰਦੇ ਹੋ ਬਿਨਾਂ ਏਜੰਟ ਨੂੰ ਦੁਬਾਰਾ ਲਿਖੇ।


## ਸਾਰਾਂਸ਼ ਅਤੇ ਵਧੇਰੇ ਚੈਲੇਂਜ  

ਤੁਹਾਡੇ ਕੋਲ ਹੈ:  

- ਇੱਕ Ed25519 ਕੁੰਜੀ ਜੋੜਾ ਬਣਾਇਆ।  
- ਇੱਕ ਏਜੰਟ ਟੂਲ ਕਾਲ ਲਈ ਰਸੀਦ ਤਿਆਰ ਕੀਤੀ ਅਤੇ ਦਸਤਖਤ ਕੀਤਾ।  
- ਰਸੀਦ ਨੂੰ ਆਫਲਾਈਨ ਸਿਰਫ ਜਾਰੀ ਕਰਨ ਵਾਲੇ ਦੀ ਸਰਵਜਨਿਕ ਕੁੰਜੀ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਪ੍ਰਮਾਣਿਤ ਕੀਤਾ, ਜੋ ਰਸੀਦ ਤੋਂ ਅਲੱਗ ਪ੍ਰਾਪਤ ਕੀਤੀ ਗਈ ਸੀ।  
- ਰਸੀਦ ਨਾਲ ਚਲਬਾਜ਼ੀ ਕੀਤੀ ਅਤੇ ਪ੍ਰਮਾਣਿਕਤਾ ਅਸਫਲ ਹੋਣ ਨੂੰ ਵੇਖਿਆ।  
- ਤਿੰਨ ਰਸੀਦਾਂ ਦੀ ਹੇਸ਼-ਚੇਨ ਵਾਲੀ ਲੜੀ ਬਣਾਈ।  
- ਲੜੀ ਦੇ ਦਰਮਿਆਨੀ ਹਿੱਸੇ ਨੂੰ ਚੇਨ੍ਹਾ ਸੀ ਅਤੇ ਦਸਤਖਤ ਫੇਲ੍ਹ ਅਤੇ ਲੜੀ-ਲਿੰਕ ਫੇਲ੍ਹ ਦੋਹਾਂ ਨੂੰ ਵੇਖਿਆ।  
- ਇੱਕ ਟੂਲ ਫੰਕਸ਼ਨ ਨੂੰ ਸਵੈਚਾਲਿਤ ਰਸੀਦ ਦਸਤਖਤੀ ਨਾਲ ਲਪੇਟਿਆ।  

**ਵਧੇਰੇ ਚੈਲੇਂਜ।** ਰਸੀਦ ਸਕੀਮਾ ਵਿੱਚ `request_id` ਫੀਲਡ (ਵੰਡੇ ਗਏ ਟ੍ਰੇਸਿੰਗ ਲਈ ਇੱਕ UUID) ਸ਼ਾਮਲ ਕਰੋ। `make_receipt` ਨੂੰ ਇਸ ਵਿੱਚ ਅਪਡੇਟ ਕਰੋ ਅਤੇ ਪੁਸ਼ਟੀ ਕਰੋ ਕਿ ਰਸੀਦਾਂ ਅੰਤ ਤੱਕ ਪ੍ਰਮਾਣਿਤ ਹੁੰਦੀਆਂ ਹਨ। ਫਿਰ ਦਸਤਖਤ ਕਰਨ ਤੋਂ ਬਾਅਦ ਇਸ ਫੀਲਡ ਨੂੰ ਸੋਧੋ ਅਤੇ ਵੇਖੋ ਕਿ ਪ੍ਰਮਾਣਿਕਤਾ ਅਸਫਲ ਹੁੰਦੀ ਹੈ। ਇਹ ਤੁਹਾਨੂੰ ਇਹ ਸਮਝਣ ਲਈ ਮਜਬੂਰ ਕਰਦਾ ਹੈ ਕਿ ਪ੍ਰਤੀਕ ਦਸਤਖਤ ਵਿੱਚ ਕੈਨੋਨਿਕਲ ਕੋਡਿੰਗ ਦਾ ਹਰ ਇੱਕ ਬਾਈਟ ਕਿਵੇਂ ਯੋਗਦਾਨ ਦਿੰਦਾ ਹੈ।  

**ਅਹਿਮ ਸੀਮਾ।** ਇੱਕ ਫਿਕਸ ਕੀ ਅਤੇ ਚੇਨ ਸੰਦਰਭ ਹੇਠ, ਰਸੀਦਾਂ ਦਸਤਖਤ ਦੇ ਅਟ੍ਰਿਬਿਊਸ਼ਨ, ਸਮੱਗਰੀ ਦੀ ਅਖੰਡਤਾ, ਅਤੇ ਦਰਸਾਏ ਗਏ ਪੂਰਵਜ ਨਾਲ ਲਿੰਕ ਸਾਬਿਤ ਕਰਦੀਆਂ ਹਨ। ਉਹ ਕਾਰਜਾਨਵਾਈ, ਸਰੋਤ ਸੱਚਾਈ, ਭਰੋਸੇਯੋਗ ਸਮਾਂ, ਪੂਰਨਤਾ, ਨੀਤੀ ਮੁਲਾਂਕਣ ਜਾਂ ਸਹੀਤਾ ਸਾਬਿਤ ਨਹੀਂ ਕਰਦੀਆਂ। ਰਸੀਦਾਂ ਇਕ ਬੁਨਿਆਦ ਹਨ। ਸ਼ਾਸਨ ਉਹ ਪ੍ਰਣਾਲੀ ਹੈ ਜੋ ਤੁਸੀਂ ਇਸ ਉੱਤੇ ਬਣਾਉਂਦੇ ਹੋ।  

ਇਸ ਸੀਮਾ ਨੂੰ ਧਿਆਨ ਵਿੱਚ ਰੱਖਦੇ ਹੋਏ ਦ lesson ਸਬਕ README ਫਿਰ ਤੋੜੋ। ਟੀਮਾਂ ਦੀ ਸਭ ਤੋਂ ਆਮ ਗਲਤੀ ਇਹ ਸਮਝਨਾ ਹੈ ਕਿ "ਸਾਡੇ ਕੋਲ ਰਸੀਦਾਂ ਹਨ" ਦਾ ਅਰਥ "ਸਾਨੂੰ ਸ਼ਾਸਨ ਮਿਲਿਆ ਹੈ" ਹੈ। ਐਸਾ ਨਹੀਂ ਹੈ। ਰਸੀਦਾਂ ਏਜੰਟ ਦੇ ਵਿਹਾਰ ਨੂੰ ਅਦਾਲਤੀ ਬਣਾ ਦਿੰਦੀਆਂ ਹਨ। ਉਹ ਇਸਨੂੰ ਸਹੀ ਨਹੀਂ ਬਣਾਉਂਦੀਆਂ।  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
